# 02 — Gerador de painéis cegos

**O que faz:** amostra pares de setor (universo = `pair_cache` ∪ `pair_cache_self`)
estratificados por pista × faixa de `N_Act`, cega cada par com um hash SHA-256, e
renderiza um painel PNG de 5 canais brutos por par -- sem qualquer informação de
piloto, pista, stint, setor ou ativação visível no painel.

**Consome:** `%run 00_core.ipynb`, `data/poc_threshold_validation/results/rules_in_poc.csv`
(gerado por `01_gap_closure.ipynb`).

**Produz:** `data/poc_threshold_validation/blind_map.csv` (hash -> identidade + N_Act +
0/1 por regra -- **não versionar a leitura deste arquivo antes do fim da rotulagem**),
`data/poc_threshold_validation/panels/{hash}.png` (um por par amostrado).

**Ordem de execução:** roda depois de `01_gap_closure.ipynb` (precisa de
`rules_in_poc.csv`). Com o cache parcial desta sessão (`RUN_FULL_BUILD = False` em
`00_core`), o universo de pares é pequeno (~36) e a amostra fica limitada a esse
universo real -- a lógica de amostragem/cegamento/renderização é validada com dados
reais, mas o tamanho-alvo de 180 só é atingível com o cache completo.

In [1]:
%run 00_core.ipynb

PROJECT_ROOT : C:\Users\to_fi\Documents\GitHub\Doutorado\Racing4all\Iracing
DATA_DIR     : C:\Users\to_fi\Documents\GitHub\Doutorado\Racing4all\Iracing\data\poc_threshold_validation
Tracks       : ['charlotte_roval_2025', 'summit_point']
Drivers      : {'Rodrigo': 'Driver A', 'Tomaz': 'Driver B', 'Morsinaldo': 'Driver C', 'Thallys': 'Driver D', 'Igor': 'Driver E', 'Hilton': 'Driver F'}
5 comparisons: ['B vs. A', 'C vs. B', 'D vs. B', 'E vs. B', 'F vs. B']
Self-comparison label: 'B vs. B (self)'
Descriptor functions loaded. DESC_SPEC = {'PEDAL_ACTIVE_PCT': 5.0, 'MIN_BRAKE_RUN': 5}
align_lap_by_dist / assert_pedal_scale loaded.
12 rules, 17 tunable thresholds, 2 descriptor parameters.
  MRP_DT               =     -0.05
  BRAKE_EFF_RATIO      =      0.85
  TRAIL_RATIO          =       0.7
  LEGACY_DV            =        -2
  LEGACY_BRAKE_GATE    =         5
  BRAKE_DIST_TOL       =     0.005
  OVERBRAKE_DELTA      =        15
  ENTRY_SLOW_REL       =    -0.015
  ENTRY_BRAKE_GATE     =    

OK (60,130 samples, laps 0–12)
──────────────────────────────────────────────────────────────
  Lap Validity Report  (13 laps total)
──────────────────────────────────────────────────────────────
  ✅ Valid   : 9 laps
  🏆 Fastest : Lap 8  (01:19.767)
  ❌ Invalid : 4 laps
     Lap   0  16:39.700  → LapTime=999.7s | GPS=27.0%<100%
     Lap   1  01:22.067  → IQR_outlier
     Lap   3  01:22.783  → IQR_outlier
     Lap  12  01:22.633  → CompletedPct=0.938<0.995 | GPS=94.0%<100%
──────────────────────────────────────────────────────────────
[SCALE] pedal channels confirmed on a 0-100 scale (brake max = 78.5 %)
pair_cache      : 1 comparison-stints, 18 sector pairs.
pair_cache_self : 1 comparison-stints, 18 sector pairs.
Total activations (cross-driver, current scope): 47
Total activations (self, current scope)        : 31

Cache persisted under C:\Users\to_fi\Documents\GitHub\Doutorado\Racing4all\Iracing\data\poc_threshold_validation\cache
stint_slopes / pattern_stats / scenario_summary loade

In [2]:
rules_in_poc_path = RESULTS_DIR / "rules_in_poc.csv"
if not rules_in_poc_path.exists():
    raise FileNotFoundError(
        f"{rules_in_poc_path} not found -- run 01_gap_closure.ipynb first.")
rules_in_poc = pd.read_csv(rules_in_poc_path)
LABELING_RULES = rules_in_poc.loc[rules_in_poc["in_labeling"], "Rule_ID"].tolist()
print(f"{len(LABELING_RULES)} rules enter the labeling universe: {LABELING_RULES}")

10 rules enter the labeling universe: ['BRAKING_POINT_MISMATCH', 'OVER_BRAKING_PEAK', 'AGGRESSIVE_UNSTABLE_BRAKE', 'ENTRY_OVER_SLOW', 'LATE_ROTATION', 'POOR_TRAIL_BRAKING', 'STEER_EFFICIENCY', 'THROTTLE_STEER_CONFLICT', 'LATE_THROTTLE_LOW_SPEED', 'EXIT_SPEED_LEGACY']


## 1. Amostragem

Universo = todos os pares de setor `(Track, Comparison, Stint, Sector)` de `pair_cache`
∪ `pair_cache_self`. `N_Act` = soma das regras em `LABELING_RULES` que dispararam nesse
par. Amostragem estratificada por pista × faixa de `N_Act` (0, 1, 2, ≥3), seed fixa,
garantindo ≥30 pares com `N_Act = 0` e ≥30 pares "B vs. B (self)".

In [3]:
def n_act_universe(df_act, is_self):
    keep = df_act[df_act["Rule_ID"].isin(LABELING_RULES)]
    g = (keep.groupby(["Track", "Comparison", "Stint", "Sector"])["Fired"]
         .sum().reset_index().rename(columns={"Fired": "N_Act"}))
    g["is_self"] = is_self
    return g

universe = pd.concat([
    n_act_universe(df_sector, is_self=False),
    n_act_universe(df_sector_self, is_self=True),
], ignore_index=True)
universe = universe.drop_duplicates(subset=["Track", "Comparison", "Stint", "Sector"]).reset_index(drop=True)


def n_act_bin(n):
    if n == 0:
        return "0"
    if n == 1:
        return "1"
    if n == 2:
        return "2"
    return ">=3"


universe["NActBin"] = universe["N_Act"].apply(n_act_bin)

print(f"Universe: {len(universe)} sector pairs "
      f"({int(universe['is_self'].sum())} self, {int((~universe['is_self']).sum())} cross-driver).")
display(universe.groupby(["Track", "NActBin"]).size().rename("n").reset_index())

Universe: 36 sector pairs (18 self, 18 cross-driver).


,Track,NActBin,n
0,CLT,0,4
1,CLT,1,13
2,CLT,2,8
3,CLT,>=3,11


In [4]:
TARGET_N   = 180
MIN_ZERO   = 30
MIN_SELF   = 30
SEED       = 20250813
rng_sample = np.random.default_rng(SEED)

target_n = min(TARGET_N, len(universe))
if target_n < TARGET_N:
    print(f"[WARN] universe has only {len(universe)} pairs (< target {TARGET_N}). "
          f"This is expected with the partial smoke-test cache; sampling the whole "
          f"universe instead. Re-run with the full cache (00_core, RUN_FULL_BUILD=True) "
          f"before trusting stratum sizes.")

def sample_df(df, k, seed):
    k = min(k, len(df))
    if k <= 0:
        return df.iloc[0:0]
    idx = np.random.default_rng(seed).choice(df.index.to_numpy(), size=k, replace=False)
    return df.loc[idx]

reserved_idx = pd.Index([])

self_pool = universe[universe["is_self"]]
sampled_self = sample_df(self_pool, MIN_SELF, SEED + 1)
reserved_idx = reserved_idx.union(sampled_self.index)

zero_pool_remaining = universe[(universe["N_Act"] == 0) & (~universe.index.isin(reserved_idx))]
n_zero_already = int((sampled_self["N_Act"] == 0).sum())
needed_zero = max(0, MIN_ZERO - n_zero_already)
sampled_zero = sample_df(zero_pool_remaining, needed_zero, SEED + 2)
reserved_idx = reserved_idx.union(sampled_zero.index)

remaining = universe[~universe.index.isin(reserved_idx)].copy()
n_fill = max(0, target_n - len(reserved_idx))

if n_fill > 0 and not remaining.empty:
    strata_sizes = remaining.groupby(["Track", "NActBin"]).size()
    props = (strata_sizes / strata_sizes.sum() * n_fill).round().astype(int)
    filled_parts = []
    for (track, bin_), n_take in props.items():
        pool = remaining[(remaining["Track"] == track) & (remaining["NActBin"] == bin_)]
        filled_parts.append(sample_df(pool, n_take, SEED + hash((track, bin_)) % 10000 + 3))
    sampled_fill = pd.concat(filled_parts) if filled_parts else remaining.iloc[0:0]
    # top up / trim to hit n_fill exactly if rounding under/overshot and pool allows
    if len(sampled_fill) < n_fill:
        leftover = remaining[~remaining.index.isin(sampled_fill.index)]
        extra = sample_df(leftover, n_fill - len(sampled_fill), SEED + 4)
        sampled_fill = pd.concat([sampled_fill, extra])
else:
    sampled_fill = remaining.iloc[0:0]

sample = pd.concat([sampled_self, sampled_zero, sampled_fill]).drop_duplicates(
    subset=["Track", "Comparison", "Stint", "Sector"]).reset_index(drop=True)

print(f"Sampled {len(sample)} pairs "
      f"({int(sample['is_self'].sum())} self, "
      f"{int((sample['N_Act'] == 0).sum())} with N_Act = 0).")

strata_table = sample.groupby(["Track", "NActBin"]).size().rename("n").reset_index()
print("\nStrata table (sampled):")
display(strata_table)

[WARN] universe has only 36 pairs (< target 180). This is expected with the partial smoke-test cache; sampling the whole universe instead. Re-run with the full cache (00_core, RUN_FULL_BUILD=True) before trusting stratum sizes.
Sampled 36 pairs (18 self, 4 with N_Act = 0).

Strata table (sampled):


,Track,NActBin,n
0,CLT,0,4
1,CLT,1,13
2,CLT,2,8
3,CLT,>=3,11


## 2. Cegamento

`hash = sha256(f"{track}|{comparison}|{stint}|{sector}|{SALT}")[:10]`. `blind_map.csv`
é o único arquivo que liga hash a identidade -- **não deve ser aberto pelo avaliador
antes do fim da rotulagem** (ver `README.md`).

In [5]:
import hashlib

SALT = "poc_threshold_validation_v1"  # fixed -- do not change between runs, or
                                       # existing panel filenames stop matching blind_map.csv

def blind_hash(track, comparison, stint, sector):
    raw = f"{track}|{comparison}|{stint}|{sector}|{SALT}"
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()[:10]

sample = sample.copy()
sample["hash"] = sample.apply(
    lambda r: blind_hash(r["Track"], r["Comparison"], r["Stint"], r["Sector"]), axis=1)

assert sample["hash"].is_unique, "hash collision in sampled set -- investigate SALT/inputs"

rule_cols = {}
for rid in LABELING_RULES:
    fired_lookup = (pd.concat([df_sector, df_sector_self])
                    .query("Rule_ID == @rid")
                    .set_index(["Track", "Comparison", "Stint", "Sector"])["Fired"])
    rule_cols[rid] = sample.set_index(["Track", "Comparison", "Stint", "Sector"]).index.map(fired_lookup).fillna(0).astype(int)

blind_map = sample[["hash", "Track", "Comparison", "Stint", "Sector", "N_Act"]].copy()
for rid, col in rule_cols.items():
    blind_map[rid] = col.values

blind_map = blind_map.sort_values("hash").reset_index(drop=True)
blind_map.to_csv(DATA_DIR / "blind_map.csv", index=False)

print(f"Saved -> {DATA_DIR / 'blind_map.csv'} ({len(blind_map)} rows)")
display(blind_map.head())

Saved -> C:\Users\to_fi\Documents\GitHub\Doutorado\Racing4all\Iracing\data\poc_threshold_validation\blind_map.csv (36 rows)


,hash,Track,Comparison,Stint,Sector,N_Act,BRAKING_POINT_MISMATCH,OVER_BRAKING_PEAK,AGGRESSIVE_UNSTABLE_BRAKE,ENTRY_OVER_SLOW,LATE_ROTATION,POOR_TRAIL_BRAKING,STEER_EFFICIENCY,THROTTLE_STEER_CONFLICT,LATE_THROTTLE_LOW_SPEED,EXIT_SPEED_LEGACY
0,06d625b7a1,CLT,B vs. B (self),stint_1,4,0,0,0,0,0,0,0,0,0,0,0
1,0a8083f0a6,CLT,C vs. B,stint_1,9,0,0,0,0,0,0,0,0,0,0,0
2,0f842174b6,CLT,C vs. B,stint_1,10,2,0,0,1,1,0,0,0,0,0,0
3,11e8539e88,CLT,B vs. B (self),stint_1,17,1,0,0,0,0,1,0,0,0,0,0
4,1f39da568d,CLT,C vs. B,stint_1,14,2,0,0,0,0,1,1,0,0,0,0


## 3. Painéis

5 subplots empilhados por par (speed km/h, brake %, throttle %, steering rad, yaw rate
rad/s), eixo x = `LapDistPct` do setor. Volta do avaliado em linha cheia, referência
tracejada cinza. Marcadores verticais em `t(Vmin)` e `t(MRP)` (posição em `LapDistPct`)
para as duas voltas. Título = apenas o hash. Nenhuma outra informação no painel.

In [6]:
SHORT_TO_TRACK = {v: k for k, v in TRACK_SHORT.items()}

# (TrackShort, Comparison, Stint) -> (track_full, ref_driver, ref_stint, test_driver, test_stint)
PAIR_DRIVER_LOOKUP = {}
for track, tshort, label, stint, ref_d, ref_s, test_d in PLAN:
    PAIR_DRIVER_LOOKUP[(tshort, label, stint)] = (track, ref_d, ref_s, test_d, stint)
for track_full in DATASETS:
    tshort = TRACK_SHORT[track_full]
    if SELF["test"] not in DATASETS[track_full]["sessions"] or SELF["ref"] not in DATASETS[track_full]["sessions"]:
        continue
    for stint in sorted(k for k in DATASETS[track_full]["sessions"][SELF["test"]] if k not in EXCLUDE_STINTS):
        PAIR_DRIVER_LOOKUP[(tshort, SELF["label"], stint)] = (
            track_full, SELF["ref"], SELF["ref_stint"], SELF["test"], stint)


def sector_slice(interp, edges, sector):
    lap_dist = interp["LapDistPct"]
    lo, hi = edges[sector], edges[sector + 1]
    mask = (lap_dist >= lo) & (lap_dist < hi) if sector < len(edges) - 2 else (lap_dist >= lo) & (lap_dist <= hi)
    idx = np.flatnonzero(mask)
    if idx.size == 0:
        return None
    sl = slice(idx[0], idx[-1] + 1)
    return {k: v[sl] for k, v in interp.items()}


def vmin_mrp_pct(sec):
    idx_vmin = int(np.argmin(sec["speed"]))
    idx_mrp  = int(np.argmax(np.abs(sec["YawRate"])))
    return float(sec["LapDistPct"][idx_vmin]), float(sec["LapDistPct"][idx_mrp])


CHANNELS = [
    ("speed",              "speed (km/h)"),
    ("brake",              "brake (%)"),
    ("throttle",           "throttle (%)"),
    ("SteeringWheelAngle", "steering (rad)"),
    ("YawRate",            "yaw rate (rad/s)"),
]

rendered_titles = []
missing = []

for _, row in sample.iterrows():
    key = (row["Track"], row["Comparison"], row["Stint"])
    if key not in PAIR_DRIVER_LOOKUP:
        missing.append(key)
        continue
    track_full, ref_d, ref_s, test_d, test_s = PAIR_DRIVER_LOOKUP[key]
    if (track_full, ref_d, ref_s) not in interp_cache or (track_full, test_d, test_s) not in interp_cache:
        missing.append(key)
        continue
    edges = TRACK_CONFIGS[track_full]["custom_edges"]
    sector = int(row["Sector"])
    sec_ref = sector_slice(interp_cache[(track_full, ref_d, ref_s)][0], edges, sector)
    sec_test = sector_slice(interp_cache[(track_full, test_d, test_s)][0], edges, sector)
    if sec_ref is None or sec_test is None:
        missing.append(key)
        continue

    vmin_pct_ref, mrp_pct_ref = vmin_mrp_pct(sec_ref)
    vmin_pct_test, mrp_pct_test = vmin_mrp_pct(sec_test)

    fig, axes = plt.subplots(len(CHANNELS), 1, figsize=(7, 10), sharex=True)
    for ax, (ch, ylabel) in zip(axes, CHANNELS):
        ax.plot(sec_test["LapDistPct"], sec_test[ch], color="tab:blue", lw=1.3, label="evaluated")
        ax.plot(sec_ref["LapDistPct"], sec_ref[ch], color="grey", lw=1.1, ls="--", label="reference")
        ax.axvline(vmin_pct_test, color="tab:blue", lw=0.8, ls="-", alpha=0.6)
        ax.axvline(mrp_pct_test,  color="tab:blue", lw=0.8, ls=":", alpha=0.6)
        ax.axvline(vmin_pct_ref,  color="grey",     lw=0.8, ls="-", alpha=0.6)
        ax.axvline(mrp_pct_ref,   color="grey",     lw=0.8, ls=":", alpha=0.6)
        ax.set_ylabel(ylabel, fontsize=8)
        ax.tick_params(labelsize=7)
    axes[-1].set_xlabel("LapDistPct (sector-local)", fontsize=8)
    axes[0].legend(fontsize=7, frameon=False, loc="upper right")

    title = row["hash"]
    fig.suptitle(title, fontsize=10)
    rendered_titles.append(title)
    fig.tight_layout(rect=[0, 0, 1, 0.98])
    fig.savefig(PANELS_DIR / f"{title}.png", dpi=130, bbox_inches="tight")
    plt.close(fig)

print(f"Rendered {len(rendered_titles)} panels -> {PANELS_DIR}")
if missing:
    print(f"[WARN] {len(missing)} sampled pairs skipped (driver/stint not in interp_cache -- "
          f"expected with the partial smoke-test cache): {missing[:5]}{' ...' if len(missing) > 5 else ''}")

Rendered 36 panels -> C:\Users\to_fi\Documents\GitHub\Doutorado\Racing4all\Iracing\data\poc_threshold_validation\panels


## 4. Checagem final

Verifica que todo PNG amostrado existe e que nenhum título usado contém strings de
`DRIVER_ALIAS` ou `TRACK_SHORT`.

In [7]:
identity_strings = set(DRIVER_ALIAS.keys()) | set(DRIVER_ALIAS.values()) | set(TRACK_SHORT.keys()) | set(TRACK_SHORT.values())

leaked = [t for t in rendered_titles if any(s.lower() in t.lower() for s in identity_strings)]
assert not leaked, f"Identity leak in panel titles: {leaked}"

missing_files = [t for t in rendered_titles if not (PANELS_DIR / f"{t}.png").exists()]
assert not missing_files, f"Missing PNG files for hashes: {missing_files}"

print(f"OK: {len(rendered_titles)} panels rendered, all titles are bare hashes, "
      f"no DRIVER_ALIAS/TRACK_SHORT string found in any title.")
print(f"OK: all {len(rendered_titles)} PNG files exist on disk under {PANELS_DIR}.")
if len(rendered_titles) < len(sample):
    print(f"[NOTE] {len(sample) - len(rendered_titles)} sampled pairs have no panel yet "
          f"(interp_cache gap) -- resolve by running 00_core with RUN_FULL_BUILD = True "
          f"and re-running this notebook.")

OK: 36 panels rendered, all titles are bare hashes, no DRIVER_ALIAS/TRACK_SHORT string found in any title.
OK: all 36 PNG files exist on disk under C:\Users\to_fi\Documents\GitHub\Doutorado\Racing4all\Iracing\data\poc_threshold_validation\panels.
